 * pure Python
 * implement decision tree and regression tree with Gini index, max depth, min number of data points
 * comparison with sklearn
 * visualization of the decision path
 * analysis of hyperparameters impact in sklearn
 * linear trees    

## Set up of data

We're taking the classic Titanic dataset

We did have to clean the data set by removing all unique columns

In [1]:
import numpy as np
import pandas as pd
import math
titanic = pd.read_csv('Titanic-Dataset.csv')

titanic.head()

# remove unique columns
titanic = titanic.drop(["PassengerId", "Name", "Cabin", "Ticket"], axis =1)

# change string columns to numbers
sex = {"female": 1, "male": 0}
embark = {"S": 0, "C": 1, "Q": -1}

titanic["Sex"] = titanic["Sex"].map(sex)
titanic["Embarked"] = titanic["Embarked"].map(embark)

# changing the name

data = titanic

data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,0.0
1,1,1,1,38.0,1,0,71.2833,1.0
2,1,3,1,26.0,0,0,7.9250,0.0
3,1,1,1,35.0,1,0,53.1000,0.0
4,0,3,0,35.0,0,0,8.0500,0.0


We will now set up the features vs the targets.

In [2]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
catagorical = ["Pclass", "Sex","SibSp", "Parch","Embarked"]


# split into training and testing 80/20?

## Gini index

$Gini=1−\sum_{i=1}^n ​p_i^2$

$Entropy=−\sum_{i=1}^n ​p_i\log_2(p_i)$

![alt text](images/image.png)

In [3]:
def gini(target):
    # split the target into survied and perished and count them
    elements, count = np.unique(target, return_counts=True)

    gini = 0
    total = len(target)
    
    for i in range(len(elements)):
        prob = count[i]/total
        gini += prob**2
    
    return 1-gini

def entropy(col):
    # calculate the unique elements and count them
    elements, count = np.unique(col, return_counts=True)

    entropy = 0
    total = len(col)

    for i in range(len(elements)):
        prob = count[i]/total
        entropy += prob * math.log2(prob)
    
    return - entropy


Dec tree

In [4]:
def weighted_gini_cat(col: str, target: str, data: pd.DataFrame):
    # col = name of columnn you might want to split on
    # target = the target: here "survied"
    # data = the entire data set

    # catagorical values in the target column
    catagories = np.unique(data[col])

    weight = 0

    # catagory with lowest gini index [cat, gini]
    min_cat = [0 , 100.0]
               
    for cat in catagories:
        cat_data = data[data[col]== cat]

        # calculate the weighted gini (gini * instances/total length)
        weight_gini = gini(cat_data[target]) * (cat_data.size/data[target].size) 
        
        if weight_gini < min_cat[1]:
            min_cat[0] = cat
            min_cat[1] = weight_gini
        
        weight += weight_gini 

    return weight, min_cat[0], min_cat[1]

def weighted_gini_num(col: str, target: str, data: pd.DataFrame):
    # col = name of columnn you might want to split on
    # target = the target: here "survied"
    # data = the entire data set

    # sort by values
    new_data = data.sort_values(col)
    new_data = new_data[col].array

    # find the averages
    averages = list()

    for i in range(new_data.size-1):
        averages.append((new_data[i]+new_data[i+1])/2)

    # average with lowest gini index [avg, gini]
    min_avg = [0 , 100.0]

    weight = 0


    for avg in averages:

        num_data = data[data[col] > avg]
    
        # calculate the weighted gini (gini * instances/total length)
        weight_gini = gini(num_data[target]) * (num_data.size/data[target].size) 

        if weight_gini < min_avg[1]:
            min_avg[0] = avg
            min_avg[1] = weight_gini
        
        weight += weight_gini
    
    return weight, min_avg[0], min_avg[1]


In [5]:
def find_split(data: pd.DataFrame, features: list, cat: list, target: str):
    smallest = {}

    # go through all features to find the one with the lowest gini
    for feat in features:
        if feat in cat:
            gini, condition, min_gini = weighted_gini_cat(feat, target, data)
        else:
            gini, condition, min_gini = weighted_gini_num(feat, target, data)

        if smallest:
            if smallest["gini"]> gini:
                smallest = {"feat": feat, "gini": gini, "condition": condition, "min_gini": min_gini}
        else: 
            smallest = {"feat": feat, "gini": gini, "condition": condition, "min_gini": min_gini}

    return smallest

def data_split(data : pd.DataFrame, feature: str, cat:bool, condition):
    # returns 2 data frames: one that satifies the condition and the other that doesnt
    # child data1 and child data2
    child_data1 = None
    child_data2 = None
    if cat:
        child_data1 = data[data[feature]== condition]
        child_data2 = data[data[feature]!= condition]

    else:
        child_data1 = data[data[feature] > condition]
        child_data2 = data[data[feature] <= condition]
    return child_data1, child_data2


In [24]:
import random 

class Graph_Node:
    def __init__(self):
        self.id = random.randrange(100, 999)

    def print(self):
        print(f"ID: {self.id}")

    def resolve(self, x=None):
        return None

    

class Node(Graph_Node):
    def __init__(self, cat: bool, condition, gini, feature: str):
        # if its catigorical, split condition and the gini index of that split
        self.cat = cat
        self.condition = condition
        self.gini = gini
        self.feature = feature

        self.id = random.randrange(100, 999)

    child1 : Graph_Node
    child2 : Graph_Node


    def print(self):
        print(f"ID: {self.id}")
        print(f"Feature: {self.feature}")
        print(f"Condition : {self.condition}")
        print(f"Gini Index : {self.gini}\n")

        print(f"First child :  ({self.id})")
        self.child1.print()

        print(f"\nSecond child :  ({self.id})")
        self.child2.print()

    def set_children(self, child1: Graph_Node, child2: Graph_Node):
        self.child1 = child1
        self.child2 = child2

    def get_child1(self):
        return self.child1
    
    def get_child2(self):
        return self.child2

    def get_if_split(self, x):
        # returns true for child 1 and false for child 2
        if self.cat:
            return x[self.feature] == self.condition
        else:
            return x[self.feature] > self.condition

    def resolve(self, x):
        if self.get_if_split(x):
            return self.child1.resolve(x)
        return self.child2.resolve(x)



class Leaf(Graph_Node):
    def __init__(self, target_values: pd.Series):
        self.target_values = target_values
        self.id = random.randrange(100, 999)

    def print(self):
        print(f"Leaf : ({self.id})")
        print(f"Resolve : {self.resolve()}")

    def resolve(self, x = None):
        # majority rule
        count = self.target_values.value_counts().to_dict()
        if 0 in count:
            if count[0] >= (1/2):
                return 0

        return 1

In [18]:
class Tree:
    def __init__(self, data: pd.DataFrame, features: list, cat: list, target: str, max_depth : int):
        self.data = data
        self.features = features
        self.cat = cat
        self.target = target   
        self.max_depth = max_depth 

    current_depth = 0
    root : Node

    def get_root(self):
        return self.root

    def build_tree(self):
        # build first node by finding the best split
        split = find_split(self.data, self.features, self.cat, self.target)
        # {"feat": feat, "gini": gini, "condition": condition}

        self.root = Node(split["feat"] in self.cat, split["condition"], split["min_gini"], split["feat"])

        child_data_1, child_data_2 = data_split(self.data, split["feat"], split["feat"] in self.cat,split["condition"])


        self.current_depth +=1

        # build the other nodes in while loop till max depth
        self.root.set_children(self.build_nodes(child_data_1),
                               self.build_nodes(child_data_2))

    def build_nodes(self, data):
        # check end conditions
        if (self.current_depth >= self.max_depth or data[self.target].nunique() == 1):
            return Leaf(data[self.target])

        split = find_split(data, self.features, self.cat, self.target)
        # {"feat": feat, "gini": gini, "condition": condition}

        parent = Node(split["feat"] in self.cat, split["condition"], split["min_gini"], split["feat"])

        child_data_1, child_data_2 = data_split(data, split["feat"], split["feat"] in self.cat,split["condition"])

        self.current_depth +=1

        # build the other nodes in while loop till max depth
        parent.set_children(self.build_nodes(child_data_1),
                               self.build_nodes(child_data_2))

        return parent
        

    def predict(self, new_data: pd.DataFrame):
        targets = list()

        # go through new data row by row and node by node
        for index, row in new_data.iterrows():
            item = row.to_dict()
            targets.append(self.root.resolve(item))

        targets = pd.Series(targets, name = "Target")

        return pd.concat([new_data, targets], axis = 1)


    def print(self):
        self.root.print()

In [43]:
titanic_tree = Tree(data, features, catagorical, "Survived", 50)
titanic_tree.build_tree()
titanic_tree.print() 

ID: 515
Feature: Sex
Condition : 1
Gini Index : 1.079328314997105

First child :  (515)
ID: 128
Feature: Pclass
Condition : 1
Gini Index : 0.14798753218593286

First child :  (128)
ID: 869
Feature: Parch
Condition : 1
Gini Index : 0.0

First child :  (869)
Leaf : (989)
Resolve : 1

Second child :  (869)
ID: 605
Feature: Parch
Condition : 0
Gini Index : 0.20454545454545456

First child :  (605)
ID: 455
Feature: Embarked
Condition : -1.0
Gini Index : 0.0

First child :  (455)
Leaf : (231)
Resolve : 1

Second child :  (455)
ID: 488
Feature: Embarked
Condition : 0.0
Gini Index : 0.0

First child :  (488)
Leaf : (984)
Resolve : 1

Second child :  (488)
ID: 880
Feature: SibSp
Condition : 1
Gini Index : 0.0

First child :  (880)
Leaf : (278)
Resolve : 1

Second child :  (880)
ID: 864
Feature: Embarked
Condition : nan
Gini Index : 0.0

First child :  (864)
ID: 731
Feature: Pclass
Condition : 0
Gini Index : 100.0

First child :  (731)
ID: 577
Feature: Pclass
Condition : 0
Gini Index : 100.0

Fi

## Prediction

In [37]:
new_data = titanic.iloc[25:,:]

new_data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
25,1,3,1,38.0,1,5,31.3875,0.0
26,0,3,0,NaN,0,0,7.2250,1.0
27,0,1,0,19.0,3,2,263.0000,0.0
28,1,3,1,NaN,0,0,7.8792,-1.0
29,0,3,0,NaN,0,0,7.8958,0.0


In [40]:
def compare(df: pd.DataFrame, target : str, predicted : str):
    df["compare"] = np.where(df[target] == df[predicted], "Success", "Fail")

    return df["compare"].value_counts().to_dict()

    

In [45]:
prediction = titanic_tree.predict(data)


comparison = compare(prediction, "Survived", "Target")

print(comparison)


{'Success': 610, 'Fail': 281}


## Visulisation of depth here

Regression Tree